# Script 1 — Ingestão, Validação Temporal & EDA
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Este notebook é o **primeiro passo do pipeline**. Ele lê os arquivos brutos da CVM,
valida e limpa os dados, calcula os KPIs financeiros e salva o resultado pronto
para o Script 2 (preparação para ML).

---

## O que este notebook faz, em ordem:

| Etapa | O que faz |
|-------|-----------|
| 0 | Configura bibliotecas, logging e parâmetros globais |
| 1A | Define as 25 empresas âncora e seus CNPJs |
| 1B | Inspeciona a estrutura dos ZIPs antes de processar |
| 2 | Lê os ZIPs de DFP **e** ITR da pasta `TCC_dados/`, converte datas imediatamente |
| 3 | Valida qualidade temporal (datas futuras, inconsistências, intervalos) |
| 4A | Transforma formato longo → largo (pivot por conta contábil) |
| 4B | Extrai Depreciação & Amortização do DFC para calcular EBITDA |
| 4C | Calcula os 15 KPIs financeiros |
| 4D | Cria 8 features temporais (ano, trimestre, flags) antes do ML |
| 5A | Remove duplicatas finais com critério explícito |
| 5B | EDA em 9 blocos analíticos |
| 6 | Salva em Parquet + CSV + relatório de auditoria JSON |

---

> **Sobre DFP vs ITR:** ambos são carregados e unificados neste script com uma
> coluna `ORIGEM` indicando a fonte. O Script 2 decidirá como usar cada um
> (ex: usar apenas DFP anuais para treino, ITR para séries mais densas).


## Etapa 0 — Dependências, logging e configuração global

**O que faz:**
Importa as bibliotecas necessárias, configura o sistema de logging (que substitui
`print()` por mensagens estruturadas com timestamp e nível de severidade), e define
todas as constantes que controlam o comportamento do pipeline.

O parâmetro mais importante aqui é `MODO_RIGOROSO`: quando `False` (padrão para TCC),
o pipeline continua mesmo encontrando problemas e apenas os registra. Quando `True`,
lança uma exceção se os erros ultrapassarem os limiares definidos — útil para
produção onde não se quer dados contaminados passando silenciosamente.

A constante `TZ_BRASIL` garante que todos os timestamps ao longo do pipeline usem
o fuso horário de São Paulo, evitando problemas em comparações de datas e na
serialização para Parquet.


In [1]:
import zipfile, logging, json, re
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo          # Python >= 3.9 (stdlib)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

# ── Timezone padrão Brasil ─────────────────────────────────────────────────
# Todos os timestamps do pipeline usam America/Sao_Paulo.
# Isso garante consistência em joins, serialização Parquet e comparações.
TZ_BRASIL   = ZoneInfo('America/Sao_Paulo')
AGORA_LOCAL = datetime.now(tz=TZ_BRASIL)

# ── Logging estruturado ────────────────────────────────────────────────────
# Substitui print() por logging → rastreabilidade em pesquisa e produção.
# Dois handlers: terminal (INFO+) e arquivo de log (DEBUG+).
logger = logging.getLogger('pipeline_cvm')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()  # evita handlers duplicados em re-execuções de célula

_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)

_fh = logging.FileHandler('outputs/pipeline_cvm.log', mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Caminho dos dados ──────────────────────────────────────────────────────
# CORREÇÃO: pasta correta conforme definido no projeto.
# Estrutura esperada:
#   TCC_dados/
#     dfp_cia_aberta_2022.zip
#     dfp_cia_aberta_2023.zip
#     dfp_cia_aberta_2024.zip
#     dfp_cia_aberta_2025.zip
#     itr_cia_aberta_2022.zip
#     itr_cia_aberta_2023.zip
#     ...
PASTA_DFP   = Path('TCC_dados/DFP')   # ZIPs anuais:      dfp_cia_aberta_YYYY.zip
PASTA_ITR   = Path('TCC_dados/ITR')   # ZIPs trimestrais: itr_cia_aberta_YYYY.zip
PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

# Parâmetros de leitura — formato real dos arquivos CVM
SEP_CVM      = ';'        # separador dos CSVs da CVM (não vírgula!)
ENCODING_CVM = 'latin1'   # encoding dos CSVs da CVM (não utf-8!)

# Colunas de data presentes nos arquivos CVM
COLS_DATA    = ['DT_REFER', 'DT_INI_EXERC', 'DT_FIM_EXERC']
FORMATO_DATA = '%Y-%m-%d'   # formato ISO 8601 oficial da CVM

# Limites para validação temporal de DFP (exercício anual)
DURACAO_DFP_MIN = 300   # mínimo de dias num período considerado anual
DURACAO_DFP_MAX = 400   # máximo de dias (exercícios atípicos como Raízen: abr→mar)
INTERVALO_MIN   = 300   # mínimo de dias entre dois DFPs consecutivos da empresa
INTERVALO_MAX   = 400   # máximo de dias entre dois DFPs consecutivos

# ── Fail-fast configurável (Patch 3) ──────────────────────────────────────
# False = TCC (continua e loga erros)
# True  = produção (lança ValueError se limiar for atingido)
MODO_RIGOROSO          = False
LIMIAR_ERROS_DATA_PCT  = 0.05   # >5%  datas inválidas → erro em modo rigoroso
LIMIAR_INCONSIST_PCT   = 0.10   # >10% inconsistências  → erro em modo rigoroso

# ── Acumulador de auditoria ────────────────────────────────────────────────
# Dicionário global que coleta estatísticas ao longo de todo o pipeline.
# Ao final, é salvo como JSON para rastreabilidade.
AUDITORIA: dict = {
    'zips_processados'        : [],
    'linhas_lidas_total'      : 0,
    'linhas_filtradas_anchor' : 0,
    'erros_data'              : [],
    'datas_futuras'           : [],
    'inconsistencias_ano'     : [],
    'periodos_irregulares'    : [],
    'duplicatas_removidas'    : 0,
    'registros_descartados'   : 0,
    'alertas_merge_outer'     : [],
}

logger.info("Pipeline iniciado | %s | Modo rigoroso: %s",
            AGORA_LOCAL.strftime('%Y-%m-%d %H:%M:%S %Z'), MODO_RIGOROSO)
logger.info("Pasta DFP : %s", PASTA_DFP.resolve())
logger.info("Pasta ITR : %s", PASTA_ITR.resolve())
logger.info("Pasta saída: %s", PASTA_SAIDA.resolve())


2026-04-07 11:22:46 | INFO     | Pipeline iniciado | 2026-04-07 11:22:46 -03 | Modo rigoroso: False
2026-04-07 11:22:46 | INFO     | Pasta DFP : C:\Users\marce\OneDrive\Área de Trabalho\TCC\TCC\TCC_dados\DFP
2026-04-07 11:22:46 | INFO     | Pasta ITR : C:\Users\marce\OneDrive\Área de Trabalho\TCC\TCC\TCC_dados\ITR
2026-04-07 11:22:46 | INFO     | Pasta saída: C:\Users\marce\OneDrive\Área de Trabalho\TCC\TCC\outputs


## Etapa 1A — Catálogo das 25 empresas âncora

**O que faz:**
Define o catálogo das 25 empresas que serão estudadas, organizadas em 5 setores.
Para cada empresa, armazena o CNPJ formatado (como aparece nos arquivos da CVM).

Em seguida, cria dicionários de consulta rápida que permitem, dado um CNPJ,
saber o nome da empresa e o setor. Isso é usado durante a leitura para filtrar
apenas os registros das 25 empresas e enriquecer o dataset com essas informações.

A normalização do CNPJ (remover pontos, barras e traços) é necessária porque
a CVM às vezes usa formatos ligeiramente diferentes entre arquivos e anos.

**Correção A aplicada:** a Vibra Energia estava com o CNPJ errado (igual ao da
Raízen). O CNPJ correto é `04.628.902/0001-38`.


In [2]:
# ── 25 Empresas âncora — 5 setores × 5 empresas ──────────────────────────
EMPRESAS = {
    'Petróleo': {
        'Petrobras':     '33.000.167/0001-01',
        'Prio':          '10.629.105/0001-68',
        'Ultrapar':      '33.256.439/0001-39',
        'Raízen':        '33.453.598/0001-23',
        'Vibra Energia': '04.628.902/0001-38',   # CORREÇÃO A: CNPJ real da Vibra
    },
    'Energia': {
        'Engie Brasil':       '02.726.168/0001-97',
        'Equatorial Energia': '02.722.865/0001-82',
        'Taesa':              '07.859.971/0001-30',
        'CPFL Energia':       '02.429.144/0001-93',
        'ISA CTEEP':          '02.998.611/0001-04',
    },
    'Varejo': {
        'Lojas Renner':  '92.754.738/0001-62',
        'Magazine Luiza':'47.960.950/0001-21',
        'Alpargatas':    '61.079.117/0001-05',
        'Arezzo':        '16.590.234/0001-76',
        'Grupo Mateus':  '01.884.051/0001-92',
    },
    'Commodities': {
        'Vale':          '33.592.510/0001-54',
        'Suzano':        '16.404.287/0001-55',
        'Klabin':        '89.637.490/0001-45',
        'Gerdau':        '33.611.500/0001-19',
        'CSN Mineração': '33.042.730/0001-04',
    },
    'Tecnologia': {
        'WEG':       '84.429.695/0001-11',
        'Totvs':     '53.113.791/0001-22',
        'Positivo':  '81.243.735/0001-48',
        'Intelbras': '82.901.000/0001-27',
        'Brisanet':  '24.047.792/0001-60',
    },
}

def normalizar_cnpj(cnpj: str) -> str:
    """Remove pontuação: '33.000.167/0001-01' → '33000167000101'"""
    return re.sub(r'\D', '', str(cnpj))

# Índices de consulta rápida (CNPJ normalizado → nome e setor)
cnpjnorm_para_nome  : dict[str, str] = {}
cnpjnorm_para_setor : dict[str, str] = {}

for setor, emps in EMPRESAS.items():
    for nome, cnpj in emps.items():
        cnorm = normalizar_cnpj(cnpj)
        cnpjnorm_para_nome[cnorm]  = nome
        cnpjnorm_para_setor[cnorm] = setor

TODOS_CNPJS_NORM = set(cnpjnorm_para_nome.keys())

logger.info("Empresas âncora: %d empresas em %d setores",
            len(cnpjnorm_para_nome), len(EMPRESAS))
for setor, emps in EMPRESAS.items():
    logger.info("  %-15s %s", setor, list(emps.keys()))


2026-04-07 11:22:57 | INFO     | Empresas âncora: 25 empresas em 5 setores
2026-04-07 11:22:57 | INFO     |   Petróleo        ['Petrobras', 'Prio', 'Ultrapar', 'Raízen', 'Vibra Energia']
2026-04-07 11:22:57 | INFO     |   Energia         ['Engie Brasil', 'Equatorial Energia', 'Taesa', 'CPFL Energia', 'ISA CTEEP']
2026-04-07 11:22:57 | INFO     |   Varejo          ['Lojas Renner', 'Magazine Luiza', 'Alpargatas', 'Arezzo', 'Grupo Mateus']
2026-04-07 11:22:57 | INFO     |   Commodities     ['Vale', 'Suzano', 'Klabin', 'Gerdau', 'CSN Mineração']
2026-04-07 11:22:57 | INFO     |   Tecnologia      ['WEG', 'Totvs', 'Positivo', 'Intelbras', 'Brisanet']


## Etapa 1B — Inspeção de estrutura dos ZIPs (Patch 1)

**O que faz:**
Antes de processar qualquer dado, esta etapa abre o ZIP mais recente disponível
e exibe um resumo da estrutura de cada arquivo CSV: quais colunas existem, seus
tipos, quantos valores nulos há e uma amostra de 3 linhas.

Por que isso é útil:
- **Para o TCC:** documenta o formato bruto dos dados da CVM, algo que a banca
  valorizará como evidência de rigor metodológico.
- **Para o código:** a CVM já mudou nomes de colunas entre versões ao longo dos
  anos. Se isso acontecer, você verá aqui antes que quebre o pipeline.
- **Para depuração:** se algo der errado no processamento, você pode voltar aqui
  e verificar se o problema estava nos dados brutos.

A função lê apenas as primeiras 200 linhas de cada CSV para ser rápida — ela
não carrega o arquivo completo.


In [3]:
def inspecionar_zip(caminho_zip: Path, max_csvs: int = 4) -> None:
    """
    Exibe a estrutura de um ZIP CVM sem carregar os dados completos.
    Mostra: lista de arquivos, colunas + tipos, cardinalidade e amostra.
    """
    print(f"\n{'='*70}")
    print(f"  📦 ZIP: {caminho_zip.name}")
    print(f"{'='*70}")

    with zipfile.ZipFile(caminho_zip) as z:
        csvs = sorted([n for n in z.namelist() if n.endswith('.csv')])
        print(f"  Arquivos CSV encontrados: {len(csvs)}")

        for nome in csvs[:max_csvs]:
            print(f"\n  📄 {nome}")
            with z.open(nome) as f:
                amostra = pd.read_csv(
                    f, sep=SEP_CVM, encoding=ENCODING_CVM,
                    nrows=200, low_memory=False, dtype=str,
                )
            print(f"     Colunas ({len(amostra.columns)}):")
            for col in amostra.columns:
                n_nulos = amostra[col].isna().sum()
                n_uniq  = amostra[col].nunique()
                print(f"       {col:<30} nulos:{n_nulos:>3}  únicos:{n_uniq:>5}")
            print(f"\n     Amostra (3 linhas):")
            print(amostra.head(3).to_string(index=False))

        if len(csvs) > max_csvs:
            print(f"\n  ... e mais {len(csvs) - max_csvs} arquivos não exibidos.")

# Executa apenas no ZIP mais recente disponível (representativo)
zips_dfp = sorted(PASTA_DFP.glob('dfp_cia_aberta_*.zip'))
zips_itr = sorted(PASTA_ITR.glob('itr_cia_aberta_*.zip'))
todos_zips = zips_dfp + zips_itr

if todos_zips:
    inspecionar_zip(todos_zips[-1], max_csvs=3)
    logger.info("ZIPs DFP disponíveis: %d | ITR disponíveis: %d",
                len(zips_dfp), len(zips_itr))
else:
    logger.warning("Nenhum ZIP encontrado. Verifique TCC_dados/DFP/ e TCC_dados/ITR/")


2026-04-07 11:23:10 | INFO     | ZIPs DFP disponíveis: 5 | ITR disponíveis: 5



  📦 ZIP: itr_cia_aberta_2025.zip
  Arquivos CSV encontrados: 19

  📄 itr_cia_aberta_2025.csv
     Colunas (9):
       CNPJ_CIA                       nulos:  0  únicos:   67
       DT_REFER                       nulos:  0  únicos:    3
       VERSAO                         nulos:  0  únicos:    3
       DENOM_CIA                      nulos:  0  únicos:   67
       CD_CVM                         nulos:  0  únicos:   67
       CATEG_DOC                      nulos:  0  únicos:    1
       ID_DOC                         nulos:  0  únicos:  200
       DT_RECEB                       nulos:  0  únicos:   43
       LINK_DOC                       nulos:  0  únicos:  200

     Amostra (3 linhas):
          CNPJ_CIA   DT_REFER VERSAO       DENOM_CIA CD_CVM CATEG_DOC ID_DOC   DT_RECEB                                                                                                              LINK_DOC
00.000.000/0001-91 2025-03-31      1 BCO BRASIL S.A. 001023       ITR 147933 2025-05-15 http://www

## Etapa 2 — Ingestão dos ZIPs de DFP e ITR

**O que faz:**
Esta é a etapa central de leitura. Ela percorre **todos** os arquivos ZIP
disponíveis na pasta `TCC_dados/` — tanto os de DFP (Demonstrações Financeiras
Padronizadas, anuais) quanto os de ITR (Informações Trimestrais) — e carrega
o demonstrativo solicitado de cada um.

**Por que unir DFP e ITR:**
- O DFP é o dado anual auditado — mais confiável, menor volume.
- O ITR é trimestral — mais granular, permite séries temporais mais densas.
- Unir os dois agora e deixar a coluna `ORIGEM` registrada permite que o
  Script 2 filtre conforme a necessidade (ex: apenas DFP para treino, ITR
  para validação cruzada temporal).

**Decisões de implementação importantes:**
1. **Datetime imediatamente após leitura** (não após merge): a função `_parse_data()`
   é chamada dentro de `ler_csv()`, antes de qualquer operação. Isso significa
   que todos os merges e filtragens usam datetime nativo — mais rápido e seguro.
2. **Sem `errors='coerce'` silencioso**: valores de data inválidos são detectados
   por máscara e registrados em `AUDITORIA['erros_data']` com arquivo, coluna
   e valor exatos.
3. **`VL_CONTA` lido como string**: evita que o pandas interprete números grandes
   em notação científica (ex: `1.23e9` ao invés de `1230000000`). A conversão
   para float é feita depois com validação explícita.
4. **Timezone Brasil aplicado**: `.tz_localize('America/Sao_Paulo')` em todo
   timestamp, evitando problemas em joins e serialização Parquet.
5. **Deduplicação de versão**: quando a CVM publica uma retificação do mesmo
   documento (VERSAO 2 substituindo VERSAO 1), mantemos apenas a versão mais alta.


In [4]:
def _parse_data(serie: pd.Series, col: str, arquivo: str) -> pd.Series:
    """
    Converte uma coluna de texto para datetime com timezone Brasil.

    Estratégia: parse vetorial com formato explícito, detecção de falhas
    por máscara (não por exceptions), registro de cada valor inválido,
    e aplicação de timezone São Paulo em todos os timestamps válidos.
    """
    resultado = pd.to_datetime(serie, format=FORMATO_DATA, errors='coerce')

    # Detecta valores que falharam no parse e os registra individualmente
    mask_inv = resultado.isna() & serie.notna() & (serie.str.strip() != '')
    if mask_inv.any():
        for idx, val in serie[mask_inv].items():
            reg = {'arquivo': arquivo, 'coluna': col, 'valor': val}
            AUDITORIA['erros_data'].append(reg)
            logger.warning("Data inválida | %s | %s | valor: %r", arquivo, col, val)

        # Fail-fast em modo rigoroso
        if MODO_RIGOROSO:
            taxa = mask_inv.sum() / max(len(serie), 1)
            if taxa > LIMIAR_ERROS_DATA_PCT:
                raise ValueError(
                    f"[MODO_RIGOROSO] {taxa:.1%} datas inválidas em {arquivo}/{col}"
                )

    # Aplica timezone Brasil — torna os timestamps timezone-aware
    # nonexistent='shift_forward': trata horas que não existem no horário de verão
    # ambiguous='infer': trata horas ambíguas na mudança de horário
    resultado = resultado.dt.tz_localize(
        TZ_BRASIL, ambiguous='infer', nonexistent='shift_forward'
    )
    return resultado


def ler_csv(zip_path: Path, nome_csv: str) -> pd.DataFrame:
    """
    Lê um CSV específico dentro de um ZIP CVM.
    Aplica: sep=';', encoding='latin1', tipos explícitos, parse de datas imediato.
    Retorna DataFrame vazio se o arquivo não existir no ZIP.
    """
    with zipfile.ZipFile(zip_path) as z:
        matches = [n for n in z.namelist() if nome_csv in n]
        if not matches:
            return pd.DataFrame()
        with z.open(matches[0]) as f:
            df = pd.read_csv(
                f,
                sep=SEP_CVM,           # CORREÇÃO B: separador explícito
                encoding=ENCODING_CVM,
                dtype={
                    'CNPJ_CIA'   : str,
                    'CD_CVM'     : str,
                    'CD_CONTA'   : str,
                    'VERSAO'     : 'Int64',
                    'VL_CONTA'   : str,    # CORREÇÃO E: str evita notação científica
                    'ORDEM_EXERC': str,
                },
                low_memory=False,
            )

    # Parse de datas imediatamente após leitura (não após merge)
    for col in COLS_DATA:
        if col in df.columns:
            df[col] = _parse_data(df[col], col, matches[0])

    # CORREÇÃO E: converte VL_CONTA com detecção explícita de não-numéricos
    if 'VL_CONTA' in df.columns:
        vl_num = pd.to_numeric(df['VL_CONTA'], errors='coerce')
        mask_inv = vl_num.isna() & df['VL_CONTA'].notna() & (df['VL_CONTA'].str.strip() != '')
        if mask_inv.any():
            logger.warning("VL_CONTA não numérico | %s | %d valores", matches[0], mask_inv.sum())
        df['VL_CONTA'] = vl_num

    return df


def carregar_demonstrativo(tipo: str, modalidade: str = 'con') -> pd.DataFrame:
    """
    Carrega um tipo de demonstrativo de TODOS os ZIPs disponíveis.
    Lê tanto arquivos DFP quanto ITR, marcando a origem de cada registro.

    Parâmetros
    ----------
    tipo      : 'DRE', 'BPA', 'BPP', 'DFC_MI', 'DFC_MD', 'DVA', 'DMPL'
    modalidade: 'con' (consolidado, preferido) ou 'ind' (individual)

    Retorna
    -------
    DataFrame longo (formato CVM) com coluna ORIGEM ('DFP' ou 'ITR'),
    filtrado para as 25 empresas âncora, datas já convertidas e tz-aware.
    """
    # Padrões de nome dos CSVs dentro dos ZIPs
    # DFP: dfp_cia_aberta_{TIPO}_{modalidade}_{ANO}.csv
    # ITR: itr_cia_aberta_{TIPO}_{modalidade}_{ANO}.csv
    fontes = [
        (sorted(PASTA_DFP.glob('dfp_cia_aberta_*.zip')), f'dfp_cia_aberta_{tipo}_{modalidade}_', 'DFP'),
        (sorted(PASTA_ITR.glob('itr_cia_aberta_*.zip')), f'itr_cia_aberta_{tipo}_{modalidade}_', 'ITR'),
    ]

    partes = []
    for zips, prefixo_csv, origem in fontes:
        if not zips:
            logger.debug("Nenhum ZIP %s encontrado na subpasta correspondente", origem)
            continue
        for zp in zips:
            df = ler_csv(zp, prefixo_csv)
            if df.empty:
                logger.debug("'%s' não encontrado em %s", prefixo_csv, zp.name)
                continue

            n_antes = len(df)
            AUDITORIA['linhas_lidas_total'] += n_antes

            # Filtra apenas as 25 empresas âncora
            df = df.copy()
            df['CNPJ_NORM'] = df['CNPJ_CIA'].apply(normalizar_cnpj)
            df = df[df['CNPJ_NORM'].isin(TODOS_CNPJS_NORM)]
            n_depois = len(df)

            AUDITORIA['linhas_filtradas_anchor'] += n_depois
            AUDITORIA['registros_descartados']   += (n_antes - n_depois)
            logger.info("  [%s] %s | %s: %d → %d linhas (âncora)",
                        origem, zp.name, tipo, n_antes, n_depois)

            if not df.empty:
                df['ORIGEM'] = origem   # 'DFP' ou 'ITR' — usado pelo Script 2
                partes.append(df)
                if zp.name not in AUDITORIA['zips_processados']:
                    AUDITORIA['zips_processados'].append(zp.name)

    if not partes:
        logger.warning("Sem dados para %s_%s em nenhum ZIP.", tipo, modalidade)
        return pd.DataFrame()

    dfinal = pd.concat(partes, ignore_index=True)

    # Enriquece com metadados das empresas âncora
    dfinal['NOME_CIA'] = dfinal['CNPJ_NORM'].map(cnpjnorm_para_nome)
    dfinal['SETOR']    = dfinal['CNPJ_NORM'].map(cnpjnorm_para_setor)
    dfinal['TIPO_DOC'] = tipo
    if 'DT_REFER' in dfinal.columns:
        dfinal['ANO_REF'] = dfinal['DT_REFER'].dt.year

    # Deduplicação de versão: mantém a retificação mais recente
    # Quando a CVM publica VERSAO 2 do mesmo documento, descarta a VERSAO 1.
    chave_v = ['CNPJ_CIA', 'DT_REFER', 'CD_CONTA', 'ORIGEM']
    if 'ORDEM_EXERC' in dfinal.columns:
        chave_v.append('ORDEM_EXERC')
    if 'VERSAO' in dfinal.columns:
        n_antes = len(dfinal)
        dfinal = (dfinal
                  .sort_values(['CNPJ_CIA', 'DT_REFER', 'VERSAO'],
                               ascending=[True, True, False])
                  .drop_duplicates(subset=chave_v, keep='first'))
        rem = n_antes - len(dfinal)
        AUDITORIA['duplicatas_removidas'] += rem
        if rem:
            logger.info("  %s: %d duplicatas de versão removidas", tipo, rem)

    anos = sorted(dfinal['ANO_REF'].dropna().astype(int).unique()) if 'ANO_REF' in dfinal.columns else []
    origens = dfinal['ORIGEM'].value_counts().to_dict() if 'ORIGEM' in dfinal.columns else {}
    logger.info("%s_%s: %d linhas | %d empresas | anos %s | origens %s",
                tipo, modalidade, len(dfinal), dfinal['NOME_CIA'].nunique(), anos, origens)
    return dfinal


## Etapa 2B — Execução do carregamento

**O que faz:**
Chama a função `carregar_demonstrativo()` para cada tipo de demonstrativo
necessário. Cada chamada varre todos os ZIPs (DFP e ITR) e retorna o dado
unificado com a coluna `ORIGEM` indicando a fonte.

Os demonstrativos carregados são:
- **DRE**: Demonstração do Resultado — receita, lucro, EBIT
- **BPA**: Balanço Patrimonial Ativo — ativos circulantes e não circulantes
- **BPP**: Balanço Patrimonial Passivo — passivos e patrimônio líquido
- **DFC_MI**: Demonstração do Fluxo de Caixa (Método Indireto) — usado para
  extrair D&A e calcular EBITDA


In [5]:
logger.info("=== Início do carregamento (DFP + ITR) ===")

dre     = carregar_demonstrativo('DRE',    'con')
bpa     = carregar_demonstrativo('BPA',    'con')
bpp     = carregar_demonstrativo('BPP',    'con')
dfc_mi  = carregar_demonstrativo('DFC_MI', 'con')
dfc_md  = carregar_demonstrativo('DFC_MD', 'con')
dva     = carregar_demonstrativo('DVA',    'con')
dmpl    = carregar_demonstrativo('DMPL',   'con')
dra     = carregar_demonstrativo('DRA',    'con')

logger.info("=== Carregamento concluído ===")
logger.info("Total lido: %d linhas | âncora: %d | erros de data: %d",
            AUDITORIA['linhas_lidas_total'],
            AUDITORIA['linhas_filtradas_anchor'],
            len(AUDITORIA['erros_data']))


2026-04-07 11:23:28 | INFO     | === Início do carregamento (DFP + ITR) ===
2026-04-07 11:23:29 | INFO     |   [DFP] dfp_cia_aberta_2021.zip | DRE: 32644 → 1482 linhas (âncora)
2026-04-07 11:23:30 | INFO     |   [DFP] dfp_cia_aberta_2022.zip | DRE: 33784 → 1488 linhas (âncora)
2026-04-07 11:23:30 | INFO     |   [DFP] dfp_cia_aberta_2023.zip | DRE: 33970 → 1494 linhas (âncora)
2026-04-07 11:23:31 | INFO     |   [DFP] dfp_cia_aberta_2024.zip | DRE: 31856 → 1504 linhas (âncora)
2026-04-07 11:23:31 | INFO     |   [DFP] dfp_cia_aberta_2025.zip | DRE: 600 → 76 linhas (âncora)
2026-04-07 11:23:33 | INFO     |   [ITR] itr_cia_aberta_2021.zip | DRE: 152686 → 7398 linhas (âncora)
2026-04-07 11:23:36 | INFO     |   [ITR] itr_cia_aberta_2022.zip | DRE: 161578 → 7234 linhas (âncora)
2026-04-07 11:23:39 | INFO     |   [ITR] itr_cia_aberta_2023.zip | DRE: 163428 → 7262 linhas (âncora)
2026-04-07 11:23:41 | INFO     |   [ITR] itr_cia_aberta_2024.zip | DRE: 164452 → 7320 linhas (âncora)
2026-04-07 11:2

## Etapa 3 — Validação da qualidade temporal

**O que faz:**
Aplica 5 verificações de qualidade sobre as datas de cada demonstrativo.
O objetivo é detectar problemas que passariam despercebidos e contaminarian
o modelo de ML.

**V1 — Datas futuras:** registros com `DT_REFER` posterior à data atual são
erros de digitação (ex: 2025 digitado como 2052). São **removidos** do dataset
e registrados no relatório de auditoria.

**V2 — Inconsistência de ano:** verifica se o ano de `DT_FIM_EXERC` bate com
`ANO_REF`. Empresas com exercício fiscal atípico (como a Raízen, que fecha em
março) terão inconsistência legítima — por isso o registro é **mantido** mas logado.

**V3 — Duração do período (Patch 2):** usa `DT_INI_EXERC` e `DT_FIM_EXERC` para
calcular quantos dias durou o exercício. Um DFP anual deve ter entre 300 e 400 dias.
Períodos fora dessa faixa são sinalizados — podem ser dados de ITR misturados ou
erros de preenchimento.

**V4 — Duplicatas:** remove registros duplicados pela chave
`(CNPJ, DT_FIM_EXERC, CD_CONTA, ORIGEM)`, mantendo o mais recente (`keep='last'`).

**V5 — Intervalos irregulares:** verifica se o espaço entre dois DFPs consecutivos
da mesma empresa está entre 300 e 400 dias. Gaps maiores (empresa sem dado por um
ano) ou menores (possível ITR passando como DFP) são sinalizados.


In [ ]:
def validar_qualidade_temporal(df: pd.DataFrame, nome: str) -> pd.DataFrame:
    """
    Executa 5 validações temporais. Retorna o DataFrame limpo.
    V1 → remove | V2, V3, V5 → loga e mantém | V4 → remove duplicatas
    """
    if df.empty:
        return df

    df = df.copy()
    n_original = len(df)
    agora_tz   = pd.Timestamp(AGORA_LOCAL)

    # ── V1: Datas futuras ────────────────────────────────────────────────
    if 'DT_REFER' in df.columns:
        mask_fut = df['DT_REFER'].notna() & (df['DT_REFER'] > agora_tz)
        n_fut = mask_fut.sum()
        if n_fut:
            exemplos = df.loc[mask_fut, ['CNPJ_CIA','DT_REFER']].head(5).to_dict('records')
            logger.warning("V1 | %s | %d datas futuras removidas | ex: %s", nome, n_fut, exemplos)
            AUDITORIA['datas_futuras'].extend(exemplos)
            df = df[~mask_fut].copy()
            AUDITORIA['registros_descartados'] += n_fut
            if MODO_RIGOROSO and (n_fut / max(n_original, 1)) > LIMIAR_INCONSIST_PCT:
                raise ValueError(f"[MODO_RIGOROSO] {n_fut} datas futuras em {nome}")

    # ── V2: DT_FIM_EXERC.year ≠ ANO_REF ─────────────────────────────────
    if 'DT_FIM_EXERC' in df.columns and 'ANO_REF' in df.columns:
        mask_v2 = (df['DT_FIM_EXERC'].notna() & df['ANO_REF'].notna() &
                   (df['DT_FIM_EXERC'].dt.year != df['ANO_REF']))
        n_v2 = mask_v2.sum()
        if n_v2:
            exemplos = df.loc[mask_v2, ['CNPJ_CIA','NOME_CIA','DT_FIM_EXERC','ANO_REF']].head(5).to_dict('records')
            logger.warning("V2 | %s | %d inconsistências ano (mantidos — podem ser fiscal year atípico) | %s",
                           nome, n_v2, exemplos)
            AUDITORIA['inconsistencias_ano'].extend(exemplos)

    # ── V3: Duração do período via DT_INI_EXERC (Patch 2) ───────────────
    if 'DT_INI_EXERC' in df.columns and 'DT_FIM_EXERC' in df.columns:
        mask_datas = df['DT_INI_EXERC'].notna() & df['DT_FIM_EXERC'].notna()
        if mask_datas.any():
            duracao = (df.loc[mask_datas,'DT_FIM_EXERC'] -
                       df.loc[mask_datas,'DT_INI_EXERC']).dt.days
            mask_anom = (duracao < DURACAO_DFP_MIN) | (duracao > DURACAO_DFP_MAX)
            if mask_anom.any():
                idx_anom = duracao[mask_anom].index
                sample   = df.loc[idx_anom[:5], ['CNPJ_CIA','NOME_CIA','DT_INI_EXERC','DT_FIM_EXERC']].copy()
                sample['duracao_dias'] = duracao[mask_anom].values[:5]
                logger.warning("V3 | %s | %d períodos com duração anômala (<300 ou >400 dias) | %s",
                               nome, mask_anom.sum(), sample.to_dict('records'))
                AUDITORIA['periodos_irregulares'].extend(sample.to_dict('records'))

    # ── V4: Duplicatas por (CNPJ, DT_FIM_EXERC, CD_CONTA, ORIGEM) ───────
    if 'DT_FIM_EXERC' in df.columns:
        chave_dup = ['CNPJ_CIA', 'DT_FIM_EXERC', 'CD_CONTA', 'ORIGEM']
        chave_dup = [c for c in chave_dup if c in df.columns]
        if 'ORDEM_EXERC' in df.columns:
            chave_dup.append('ORDEM_EXERC')
        n_ant = len(df)
        df = df.drop_duplicates(subset=chave_dup, keep='last')
        rem = n_ant - len(df)
        if rem:
            AUDITORIA['duplicatas_removidas'] += rem
            logger.info("V4 | %s | %d duplicatas removidas", nome, rem)

    # ── V5: Intervalos irregulares entre exercícios consecutivos ─────────
    if 'DT_REFER' in df.columns and 'CNPJ_CIA' in df.columns:
        df_ord = (df[['CNPJ_CIA','DT_REFER','ORIGEM']].drop_duplicates()
                  .sort_values(['CNPJ_CIA','ORIGEM','DT_REFER']))
        df_ord['_delta'] = df_ord.groupby(['CNPJ_CIA','ORIGEM'])['DT_REFER'].diff().dt.days
        irr = df_ord[df_ord['_delta'].notna() &
                     ((df_ord['_delta'] < INTERVALO_MIN) | (df_ord['_delta'] > INTERVALO_MAX))]
        if not irr.empty:
            logger.warning("V5 | %s | %d intervalos irregulares entre exercícios:\n%s",
                           nome, len(irr),
                           irr[['CNPJ_CIA','DT_REFER','ORIGEM','_delta']].to_string())

    logger.info("Validação | %-8s | %d → %d linhas (-%d)", nome,
                n_original, len(df), n_original - len(df))
    return df

logger.info("=== Validação temporal ===")
dre = validar_qualidade_temporal(dre, 'DRE')
bpa = validar_qualidade_temporal(bpa, 'BPA')
bpp = validar_qualidade_temporal(bpp, 'BPP')
dfc_mi = validar_qualidade_temporal(dfc_mi, 'DFC_MI')
dfc_md = validar_qualidade_temporal(dfc_md, 'DFC_MD')
dva = validar_qualidade_temporal(dva, 'DVA')
dmpl = validar_qualidade_temporal(dmpl, 'DMPL')
dra = validar_qualidade_temporal(dra, 'DRA')


## Etapa 4A — Pivotagem: formato longo → formato largo

**O que faz:**
Os arquivos da CVM chegam em **formato longo**: cada linha é uma conta contábil
de uma empresa em um período. Por exemplo, a Petrobras no ano 2024 tem uma linha
para a conta 3.01 (Receita), outra para 3.03 (Lucro Bruto), outra para 3.05
(EBIT) e assim por diante — dezenas de linhas para um único período.

Para calcular KPIs, precisamos que cada empresa/período seja **uma linha** com
todas as contas como colunas. Isso é feito pelo `pivot_table`.

**Filtro `ORDEM_EXERC = 'ÚLTIMO'`:** cada demonstrativo vem com duas versões
dos valores — o período atual (`ÚLTIMO`) e o período anterior para comparação
(`PENÚLTIMO`). Mantemos apenas o `ÚLTIMO` para evitar duplicar dados.

**Correção C:** antes do pivot, o DataFrame é ordenado por `VERSAO` em ordem
crescente para garantir que `aggfunc='last'` sempre pega a versão mais recente.
Sem essa ordenação, o resultado poderia variar entre execuções.

**Merge progressivo:** após pivotar cada demonstrativo separadamente, eles são
unidos por `(CNPJ_CIA, NOME_CIA, SETOR, ANO_REF, DT_REFER, ORIGEM)`. O `how='outer'`
é usado para não perder empresas que têm BPA mas não têm DRE num determinado período
— esses casos são sinalizados adiante.


In [ ]:
def pivotar(df: pd.DataFrame, sufixo: str) -> pd.DataFrame:
    """
    Transforma formato longo → largo.
    Filtra ORDEM_EXERC='ÚLTIMO', ordena por VERSAO (CORREÇÃO C),
    e renomeia colunas como '{sufixo}_{CD_CONTA}'.
    """
    if df.empty:
        return pd.DataFrame()

    # Mantém apenas o exercício corrente (não o comparativo)
    if 'ORDEM_EXERC' in df.columns:
        df = df[df['ORDEM_EXERC'] == 'ÚLTIMO'].copy()

    # CORREÇÃO C: ordena antes do pivot para resultado determinístico
    sort_cols = ['CNPJ_CIA', 'DT_REFER', 'ORIGEM']
    if 'VERSAO' in df.columns:
        sort_cols.append('VERSAO')
    df = df.sort_values(sort_cols, ascending=True)

    df['CD_CONTA'] = df['CD_CONTA'].astype(str).str.strip()
    df['VL_CONTA'] = pd.to_numeric(df['VL_CONTA'], errors='coerce')

    chave_idx = [c for c in ['CNPJ_CIA','NOME_CIA','SETOR','ANO_REF','DT_REFER','ORIGEM']
                 if c in df.columns]

    pivot = df.pivot_table(
        index=chave_idx,
        columns='CD_CONTA',
        values='VL_CONTA',
        aggfunc='last',   # determinístico após sort_values acima
    )
    pivot.columns = [f'{sufixo}_{c}' for c in pivot.columns]
    pivot.columns.name = None
    result = pivot.reset_index()
    logger.info("Pivot %-6s: %d linhas × %d colunas", sufixo, *result.shape)
    return result

logger.info("=== Pivotagem ===")
p_dre    = pivotar(dre,    'DRE')
p_bpa    = pivotar(bpa,    'BPA')
p_bpp    = pivotar(bpp,    'BPP')
p_dfc_mi = pivotar(dfc_mi, 'DFC_MI')
p_dfc_md = pivotar(dfc_md, 'DFC_MD')
p_dva    = pivotar(dva,    'DVA')
p_dmpl   = pivotar(dmpl,   'DMPL')
p_dra    = pivotar(dra,    'DRA')

# ── PATCH 1 — Anchor fixo de empresas-períodos ────────────────────────────
# PROBLEMA ORIGINAL: dataset = p_dre.copy() → empresas sem DRE no período
# são descartadas antes mesmo do primeiro merge. O outer join subsequente
# não consegue recuperá-las porque a base pivô já não as contém.
#
# SOLUÇÃO: construir um anchor com TODOS os (CNPJ_CIA, DT_REFER, ORIGEM)
# presentes em QUALQUER demonstrativo. O merge é sempre LEFT a partir desse anchor.
# Isso garante que todas as 25 empresas permanecem no dataset independentemente
# de qual demonstrativo possui ou não dados para um dado período.

CHAVE_MERGE = ['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO_REF', 'DT_REFER', 'ORIGEM']
CHAVE_ANCHOR = ['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO_REF', 'DT_REFER', 'ORIGEM']

# Coleta todos os períodos de todas as demonstrações
_frames_anchor = []
for _p, _nome in [
    (p_dre, 'DRE'), (p_bpa, 'BPA'), (p_bpp, 'BPP'),
    (p_dfc_mi, 'DFC_MI'), (p_dfc_md, 'DFC_MD'),
    (p_dva, 'DVA'), (p_dmpl, 'DMPL'), (p_dra, 'DRA'),
]:
    if _p.empty:
        continue
    cols_anchor = [c for c in CHAVE_ANCHOR if c in _p.columns]
    _frames_anchor.append(_p[cols_anchor].drop_duplicates())

if not _frames_anchor:
    logger.error("Nenhuma demonstrativo pivotado — dataset vazio.")
    dataset = pd.DataFrame()
else:
    # Anchor = união de todos os períodos de todas as demos
    anchor = pd.concat(_frames_anchor, ignore_index=True).drop_duplicates()
    logger.info("Anchor criado: %d períodos | %d empresas",
                len(anchor), anchor['CNPJ_CIA'].nunique())

    # Merge progressivo LEFT a partir do anchor (garante 25 empresas)
    dataset = anchor.copy()
    for p_df, nome_p in [
        (p_dre,    'DRE'),
        (p_bpa,    'BPA'),
        (p_bpp,    'BPP'),
        (p_dfc_mi, 'DFC_MI'),
        (p_dfc_md, 'DFC_MD'),
        (p_dva,    'DVA'),
        (p_dmpl,   'DMPL'),
        (p_dra,    'DRA'),
    ]:
        if p_df.empty:
            logger.warning("Pivot %s vazio — merge ignorado", nome_p)
            continue

        chave = [c for c in CHAVE_MERGE if c in dataset.columns and c in p_df.columns]

        n_ant = len(dataset)
        # Colunas de dados apenas (exclui chaves para evitar duplicação)
        cols_dados = [c for c in p_df.columns if c not in chave]
        dataset = dataset.merge(
            p_df[chave + cols_dados],
            on=chave,
            how='left',          # ← LEFT garante que anchor nunca perde linhas
            suffixes=(''  , f'__{nome_p}')
        )
        # Remove colunas duplicadas que possam ter sido geradas (sufixo __NOME_P)
        cols_dup = [c for c in dataset.columns if f'__{nome_p}' in c]
        if cols_dup:
            dataset.drop(columns=cols_dup, inplace=True)

        logger.info("Merge %-8s: %d → %d linhas | %d empresas",
                    nome_p, n_ant, len(dataset), dataset['CNPJ_CIA'].nunique())

    logger.info("Dataset pós-pivot: %d × %d | %d empresas",
                *dataset.shape, dataset['CNPJ_CIA'].nunique())


## Etapa 4B — Extração de D&A via DFC Método Indireto

**O que faz:**
Extrai a Depreciação & Amortização (D&A) para calcular o EBITDA.

**Por que é necessário:** o DRE padrão da CVM não tem uma linha exclusiva para D&A.
Ela está embutida no custo dos produtos (conta 3.02) e nas despesas operacionais
(3.04) junto com outros itens. A solução é buscá-la no DFC pelo Método Indireto:
nesse demonstrativo, D&A aparece como ajuste ao lucro líquido (subcontas 6.01.01.xx)
porque é uma despesa que não representa saída de caixa.

A busca é feita por palavras-chave no campo `DS_CONTA`: "deprecia", "amortiza",
"exaust" e "depletion". Os valores encontrados são somados por empresa/período.

**Limitação documentada:** empresas que publicam apenas o DFC pelo Método Direto
(DFC_MD) não têm esses ajustes explicitados — para elas, o EBITDA será `NaN`.
Isso é registrado como limitação do estudo.

D&A igual a zero é substituído por `NaN` para não contaminar o EBITDA com
zeros espúrios (zero real vs. zero por ausência de dado são situações diferentes).


In [ ]:
def extrair_dna_completo(dfc_df: pd.DataFrame, dre_con: pd.DataFrame = None) -> pd.DataFrame:
    """
    Extrai Depreciação & Amortização (D&A) com hierarquia de 3 níveis:
    1) DFC_MI por CD_CONTA 6.01.01.xx  (mais preciso — padrão CVM)
    2) DFC_MI por DS_CONTA (fallback textual)
    3) DRE consolidado por DS_CONTA (último recurso)

    PATCH 2: prioriza CD_CONTA para evitar captura incorreta de outras contas.
    A conta 6.01.01 no DFC_MI consolida Depreciação+Amortização+Exaustão.
    Subcontas 6.01.01.xx podem existir por empresa — somadas corretamente.

    Retorna DataFrame com coluna DNA por (CNPJ_CIA, ANO_REF, DT_REFER, ORIGEM).
    """

    # =================================================================
    # 1. EXTRAÇÃO VIA DFC — CD_CONTA (MÉTODO PRIMÁRIO — padrão CVM)
    # =================================================================
    def extrair_dfc_cd_conta():
        if dfc_df is None or dfc_df.empty:
            return pd.DataFrame()
        if 'CD_CONTA' not in dfc_df.columns:
            logger.warning("CD_CONTA não encontrada no DFC_MI")
            return pd.DataFrame()

        df = dfc_df.copy()
        if 'ORDEM_EXERC' in df.columns:
            df = df[df['ORDEM_EXERC'] == 'ÚLTIMO']

        # Conta 6.01.01 = D&A no padrão CVM / IFRS
        # Captura 6.01.01 exata e todas as subcontas 6.01.01.xx
        mask_cd = df['CD_CONTA'].astype(str).str.match(r'6\.01\.01(\.\d+)*$')
        df_dna = df[mask_cd].copy()

        if df_dna.empty:
            logger.debug("CD_CONTA 6.01.01 não encontrada no DFC_MI")
            return pd.DataFrame()

        df_dna['VL_CONTA'] = pd.to_numeric(df_dna['VL_CONTA'], errors='coerce').abs()

        chave = [c for c in ['CNPJ_CIA','NOME_CIA','SETOR','ANO_REF','DT_REFER','ORIGEM']
                 if c in df_dna.columns]

        result = (df_dna.groupby(chave)['VL_CONTA']
                  .sum().reset_index()
                  .rename(columns={'VL_CONTA': 'DNA_DFC_CD'}))
        result['DNA_DFC_CD'] = result['DNA_DFC_CD'].replace(0, np.nan)
        logger.info("D&A via CD_CONTA 6.01.01: %d registros | %.0f%% cobertura",
                    len(result), result['DNA_DFC_CD'].notna().mean() * 100)
        return result

    # =================================================================
    # 2. EXTRAÇÃO VIA DFC — DS_CONTA (FALLBACK TEXTUAL)
    # =================================================================
    def extrair_dfc_ds_conta():
        if dfc_df is None or dfc_df.empty:
            return pd.DataFrame()
        if 'DS_CONTA' not in dfc_df.columns:
            return pd.DataFrame()

        PALAVRAS = [
            'deprecia', 'amortiza', 'exaust', 'depletion',
            'depreciacao', 'depreciação', 'amortizacao', 'amortização'
        ]

        df = dfc_df.copy()
        if 'ORDEM_EXERC' in df.columns:
            df = df[df['ORDEM_EXERC'] == 'ÚLTIMO']

        mask = (df['DS_CONTA'].astype(str).str.lower()
                .str.contains('|'.join(PALAVRAS), na=False))

        # PATCH 2: exclui contas de investimento (6.02.xx) para não capturar
        # "aquisição de imobilizado" que pode conter "amortiza" no texto
        if 'CD_CONTA' in df.columns:
            mask_inv = df['CD_CONTA'].astype(str).str.startswith('6.02')
            mask = mask & ~mask_inv

        df_dna = df[mask].copy()
        if df_dna.empty:
            return pd.DataFrame()

        df_dna['VL_CONTA'] = pd.to_numeric(df_dna['VL_CONTA'], errors='coerce').abs()

        chave = [c for c in ['CNPJ_CIA','NOME_CIA','SETOR','ANO_REF','DT_REFER','ORIGEM']
                 if c in df_dna.columns]

        result = (df_dna.groupby(chave)['VL_CONTA']
                  .sum().reset_index()
                  .rename(columns={'VL_CONTA': 'DNA_DFC_DS'}))
        result['DNA_DFC_DS'] = result['DNA_DFC_DS'].replace(0, np.nan)
        return result

    # =================================================================
    # 3. EXTRAÇÃO VIA DRE CONSOLIDADO (ÚLTIMO RECURSO)
    # =================================================================
    def extrair_dre():
        if dre_con is None or dre_con.empty:
            return pd.DataFrame()

        col_conta = next((c for c in dre_con.columns
                          if 'conta' in c.lower() or 'descricao' in c.lower()), None)
        col_valor = next((c for c in dre_con.columns
                          if 'valor' in c.lower() or 'vl_conta' in c.lower()), None)

        if not col_conta or not col_valor:
            logger.warning("Colunas de conta/valor não encontradas na DRE para D&A")
            return pd.DataFrame()

        termos = ['deprecia','amortiza','depreciacao','depreciação',
                  'amortizacao','amortização']
        df = dre_con.copy()
        if 'ORDEM_EXERC' in df.columns:
            df = df[df['ORDEM_EXERC'] == 'ÚLTIMO']
        df[col_conta] = df[col_conta].astype(str).str.lower()
        mask = df[col_conta].apply(lambda x: any(t in x for t in termos))
        df = df[mask]
        if df.empty:
            return pd.DataFrame()

        df[col_valor] = pd.to_numeric(df[col_valor], errors='coerce').abs()
        chave = [c for c in ['CNPJ_CIA','NOME_CIA','SETOR','ANO_REF','DT_REFER','ORIGEM']
                 if c in df.columns]
        result = (df.groupby(chave)[col_valor]
                  .sum().reset_index()
                  .rename(columns={col_valor: 'DNA_DRE'}))
        result['DNA_DRE'] = result['DNA_DRE'].replace(0, np.nan)
        return result

    # =================================================================
    # 4. COMBINAÇÃO HIERÁRQUICA
    # =================================================================
    dna_cd   = extrair_dfc_cd_conta()
    dna_ds   = extrair_dfc_ds_conta()
    dna_dre  = extrair_dre()

    # Merge progressivo dos três níveis
    base = None
    for _df, _nome in [(dna_cd,'DNA_DFC_CD'),(dna_ds,'DNA_DFC_DS'),(dna_dre,'DNA_DRE')]:
        if _df.empty:
            continue
        if base is None:
            base = _df
        else:
            chave_m = [c for c in base.columns if c not in ['DNA_DFC_CD','DNA_DFC_DS','DNA_DRE']]
            base = base.merge(_df, on=chave_m, how='outer')

    if base is None:
        logger.warning("D&A não encontrado em nenhuma fonte")
        return pd.DataFrame()

    # Hierarquia: CD_CONTA > DS_CONTA > DRE
    base['DNA'] = base.get('DNA_DFC_CD')
    if 'DNA_DFC_DS' in base.columns:
        base['DNA'] = base['DNA'].fillna(base['DNA_DFC_DS'])
    if 'DNA_DRE' in base.columns:
        base['DNA'] = base['DNA'].fillna(base['DNA_DRE'])
    base['DNA'] = base['DNA'].replace(0, np.nan)

    cob_cd  = base.get('DNA_DFC_CD', pd.Series(dtype=float)).notna().mean()
    cob_ds  = base.get('DNA_DFC_DS', pd.Series(dtype=float)).notna().mean()
    cob_dre = base.get('DNA_DRE',    pd.Series(dtype=float)).notna().mean()
    cob_fin = base['DNA'].notna().mean()

    logger.info("D&A — CD_CONTA: %.0f%% | DS_CONTA: %.0f%% | DRE: %.0f%% | Final: %.0f%%",
                cob_cd*100, cob_ds*100, cob_dre*100, cob_fin*100)
    return base


dna = extrair_dna_completo(
    dfc_df=dfc_mi,
    dre_con=dre
)


In [ ]:
def extrair_capex(dfc_df: pd.DataFrame) -> pd.DataFrame:
    """
    Extrai CAPEX a partir do DFC consolidado.

    PATCH 3 — Hierarquia de extração:
    1) CD_CONTA 6.02.01.xx e 6.02.02.xx (Aquisição de Imobilizado/Intangível — padrão CVM)
    2) CD_CONTA 6.02.xx genérico (qualquer conta de investimento)
    3) DS_CONTA fallback (apenas em contas 6.02.xx, nunca 6.01.xx)

    A seção 6.02 do DFC = Atividades de Investimento.
    6.02.01 = Aquisições de imobilizado / intangível / investimentos.
    Valores são negativos no DFC (saídas de caixa) → abs() aplicado.
    """

    if dfc_df is None or dfc_df.empty:
        return pd.DataFrame()

    df = dfc_df.copy()
    if 'ORDEM_EXERC' in df.columns:
        df = df[df['ORDEM_EXERC'] == 'ÚLTIMO']

    chave = [c for c in ['CNPJ_CIA','NOME_CIA','SETOR','ANO_REF','DT_REFER','ORIGEM']
             if c in df.columns]

    capex_result = pd.DataFrame()

    # ── Método 1: CD_CONTA 6.02.01 e 6.02.02 ─────────────────────────────
    if 'CD_CONTA' in df.columns:
        mask_cd = df['CD_CONTA'].astype(str).str.match(r'6\.02\.(01|02)(\.\d+)*$')
        df_cd = df[mask_cd].copy()

        if not df_cd.empty:
            df_cd['VL_CONTA'] = pd.to_numeric(df_cd['VL_CONTA'], errors='coerce').abs()
            capex_result = (df_cd.groupby(chave)['VL_CONTA']
                            .sum().reset_index()
                            .rename(columns={'VL_CONTA': 'CAPEX'}))
            capex_result['CAPEX'] = capex_result['CAPEX'].replace(0, np.nan)
            cob = capex_result['CAPEX'].notna().mean()
            logger.info("CAPEX via CD_CONTA 6.02.01/02: %d registros | %.0f%% cobertura",
                        len(capex_result), cob * 100)

        # ── Método 2: CD_CONTA 6.02.xx genérico (se 6.02.01/02 insuficiente) ──
        empresas_sem_capex = set()
        if not capex_result.empty and 'CNPJ_CIA' in capex_result.columns:
            empresas_sem_capex = (
                set(df['CNPJ_CIA'].unique()) -
                set(capex_result[capex_result['CAPEX'].notna()]['CNPJ_CIA'].unique())
            )

        if empresas_sem_capex or capex_result.empty:
            mask_cd2 = df['CD_CONTA'].astype(str).str.match(r'6\.02(\.\d+)+$')
            df_cd2 = df[mask_cd2 & df['CNPJ_CIA'].isin(empresas_sem_capex)].copy() if empresas_sem_capex else df[mask_cd2].copy()

            if not df_cd2.empty:
                # Exclui subcontas de recebimentos (positivas no investimento = desinvestimento)
                # Mantém apenas saídas líquidas de caixa (valores negativos → abs)
                df_cd2['VL_CONTA'] = pd.to_numeric(df_cd2['VL_CONTA'], errors='coerce')
                df_cd2 = df_cd2[df_cd2['VL_CONTA'] <= 0]  # apenas saídas
                df_cd2['VL_CONTA'] = df_cd2['VL_CONTA'].abs()

                capex_cd2 = (df_cd2.groupby(chave)['VL_CONTA']
                             .sum().reset_index()
                             .rename(columns={'VL_CONTA': 'CAPEX'}))
                capex_cd2['CAPEX'] = capex_cd2['CAPEX'].replace(0, np.nan)

                if capex_result.empty:
                    capex_result = capex_cd2
                else:
                    capex_result = pd.concat([capex_result, capex_cd2], ignore_index=True)

    # ── Método 3: DS_CONTA fallback (apenas em contas 6.02.xx) ───────────
    if capex_result.empty or capex_result['CAPEX'].isna().mean() > 0.5:
        if 'DS_CONTA' in df.columns:
            PALAVRAS_CAPEX = ['imobiliz', 'intang', 'aquisi', 'compra ativo']
            mask_txt = (df['DS_CONTA'].astype(str).str.lower()
                        .str.contains('|'.join(PALAVRAS_CAPEX), na=False))
            # CRÍTICO: restringe ao grupo 6.02 para não capturar D&A de 6.01
            if 'CD_CONTA' in df.columns:
                mask_txt = mask_txt & df['CD_CONTA'].astype(str).str.startswith('6.02')

            df_txt = df[mask_txt].copy()
            if not df_txt.empty:
                df_txt['VL_CONTA'] = pd.to_numeric(df_txt['VL_CONTA'], errors='coerce').abs()
                capex_txt = (df_txt.groupby(chave)['VL_CONTA']
                             .sum().reset_index()
                             .rename(columns={'VL_CONTA': 'CAPEX'}))
                capex_txt['CAPEX'] = capex_txt['CAPEX'].replace(0, np.nan)
                if capex_result.empty:
                    capex_result = capex_txt
                    logger.info("CAPEX via DS_CONTA (fallback): %d registros", len(capex_result))

    if capex_result.empty:
        logger.warning("CAPEX não extraído — nenhum método retornou dados")
        return pd.DataFrame()

    # Deduplicar por chave (pega maior valor se houver sobreposição)
    capex_result = (capex_result.sort_values('CAPEX', ascending=False)
                    .drop_duplicates(subset=chave, keep='first')
                    .reset_index(drop=True))

    logger.info("CAPEX final: %d registros | cobertura %.0f%%",
                len(capex_result), capex_result['CAPEX'].notna().mean() * 100)
    return capex_result


# ============================================================
# CAPEX — EXTRAÇÃO E INTEGRAÇÃO
# ============================================================
capex = extrair_capex(dfc_mi)

if not capex.empty and not dataset.empty:
    chave_capex = [c for c in ['CNPJ_CIA','ANO_REF','DT_REFER','ORIGEM']
                   if c in dataset.columns and c in capex.columns]
    dataset = dataset.merge(
        capex[chave_capex + ['CAPEX']],
        on=chave_capex,
        how='left'
    )
    logger.info("CAPEX integrado ao dataset | cobertura: %.0f%%",
                dataset['CAPEX'].notna().mean() * 100)
else:
    logger.warning("CAPEX não integrado (dados insuficientes)")


## Etapa 4C — Cálculo dos 15 KPIs Financeiros

**O que faz:**
Calcula os 19 indicadores financeiros (15 originais + 4 de caixa) a partir das contas contábeis brutas.

**PATCH 4 — Correção crítica do FCO:**
A versão V4 buscava `get('DFC','6.01')` → coluna `DFC_6.01` que **nunca existe**.
O pivot gera `DFC_MI_6.01` e `DFC_MD_6.01`. Essa divergência de prefixo era a causa
raiz dos 5 KPIs de caixa zerados (`fco_receita`, `fco_lucro`, `FCF`, `margem_fcf`, `conversao_caixa`).

**Hierarquia de FCO:**
1. `DFC_MI_6.01` — conta raiz do Método Indireto (preferida)
2. `DFC_MI_6.01.01` — subconta usada por algumas empresas
3. `DFC_MD_6.01` — fallback para empresas que usam Método Direto

**Nota:** a integração do D&A (DNA) ao dataset agora ocorre aqui, antes do `calcular_kpis()`.


In [ ]:
LISTA_KPIS = [
    'margem_bruta','margem_ebit','margem_liquida','margem_ebitda',
    'roe','roa','liquidez_corrente','liquidez_imediata',
    'endividamento','alavancagem_de','div_liquida','cobertura_juros',
    'giro_ativo','fco_receita','fco_lucro','EBITDA',
    'FCF','margem_fcf','conversao_caixa'
]

def calcular_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula KPIs financeiros completos com suporte a DNA, CAPEX e FCF.

    PATCH 4 — Correção crítica do FCO:
    O pipeline gera colunas 'DFC_MI_6.01' e 'DFC_MD_6.01' (não 'DFC_6.01').
    A função get('DFC','6.01') procura 'DFC_6.01' → sempre NaN.
    Correção: buscar na ordem DFC_MI_6.01 → DFC_MI_6.01.01 → DFC_MD_6.01.
    Isso resolve 100% dos KPIs derivados de FCO zerados.
    """

    d = df.copy()

    def get(p, c):
        col = f'{p}_{c}'
        return d[col].copy() if col in d.columns else pd.Series(np.nan, index=d.index)

    def get_first(*cols):
        """Retorna a primeira coluna não-nula da lista (para fallback entre demonstrativos)."""
        result = pd.Series(np.nan, index=d.index)
        for col in cols:
            if col in d.columns:
                result = result.combine_first(d[col])
        return result

    def _div(num, den):
        return num / den.replace(0, np.nan)

    # =========================
    # EXTRAÇÃO DE CONTAS BASE
    # =========================
    receita     = get('DRE','3.01')
    lucro_bruto = get('DRE','3.03')
    ebit        = get('DRE','3.05')
    lucro_liq   = get('DRE','3.11')
    desp_fin    = get('DRE','3.06')

    ativo_tot   = get('BPA','1')
    ativo_circ  = get('BPA','1.01')
    caixa       = get('BPA','1.01.01')

    pass_circ   = get('BPP','2.01')
    div_cp      = get('BPP','2.01.04')
    div_lp      = get('BPP','2.02.01')
    pat_liq     = get('BPP','2.03')

    # ── PATCH 4: FCO com hierarquia correta ──────────────────────────────
    # O pivot gera sufixos 'DFC_MI' e 'DFC_MD' — nunca 'DFC'.
    # Hierarquia: DFC_MI_6.01 (conta raiz MI) →
    #             DFC_MI_6.01.01 (primeira subconta MI — algumas empresas estruturam assim) →
    #             DFC_MD_6.01 (método direto como último fallback)
    fco = get_first('DFC_MI_6.01', 'DFC_MI_6.01.01', 'DFC_MD_6.01')

    # Log de diagnóstico do FCO
    _fco_mi     = get('DFC_MI','6.01')   # para diagnóstico
    _fco_mi_sub = get('DFC_MI','6.01.01')
    _fco_md     = get('DFC_MD','6.01')
    logger.info("FCO cobertura — DFC_MI_6.01: %.0f%% | DFC_MI_6.01.01: %.0f%% | DFC_MD_6.01: %.0f%% | Final: %.0f%%",
                _fco_mi.notna().mean()*100,
                _fco_mi_sub.notna().mean()*100,
                _fco_md.notna().mean()*100,
                fco.notna().mean()*100)

    # =========================
    # AJUSTES (DNA e CAPEX)
    # =========================
    dna   = d['DNA'].copy()   if 'DNA'   in d.columns else pd.Series(0.0, index=d.index)
    capex = d['CAPEX'].copy() if 'CAPEX' in d.columns else pd.Series(np.nan, index=d.index)

    # =========================
    # DERIVADOS
    # =========================
    div_bruta = div_cp.fillna(0) + div_lp.fillna(0)
    ebitda    = ebit + dna.fillna(0)
    fcf       = fco - capex

    # =========================
    # KPIs
    # =========================
    d['margem_bruta']      = _div(lucro_bruto, receita)
    d['margem_ebit']       = _div(ebit, receita)
    d['margem_liquida']    = _div(lucro_liq, receita)
    d['margem_ebitda']     = _div(ebitda, receita)

    d['roe']               = _div(lucro_liq, pat_liq)
    d['roa']               = _div(lucro_liq, ativo_tot)

    d['liquidez_corrente'] = _div(ativo_circ, pass_circ)
    d['liquidez_imediata'] = _div(caixa, pass_circ)

    d['endividamento']     = _div(div_bruta, ativo_tot)
    d['alavancagem_de']    = _div(div_bruta, pat_liq)

    d['div_liquida']       = div_bruta - caixa.fillna(0)

    d['cobertura_juros']   = _div(ebit, desp_fin.abs())

    d['giro_ativo']        = _div(receita, ativo_tot)

    d['fco_receita']       = _div(fco, receita)
    d['fco_lucro']         = _div(fco, lucro_liq)

    d['EBITDA']            = ebitda

    d['FCF']               = fcf
    d['margem_fcf']        = _div(fcf, receita)
    d['conversao_caixa']   = _div(fco, ebitda)

    # =========================
    # LOG DE COBERTURA
    # =========================
    logger.info("KPIs calculados:")
    for kpi in LISTA_KPIS:
        if kpi in d.columns:
            cob   = d[kpi].notna().mean()
            nivel = "✅" if cob >= 0.5 else "⚠️"
            logger.info("  %s %-25s %.0f%%", nivel, kpi, cob * 100)

    return d


# =========================
# EXECUÇÃO
# =========================
if not dataset.empty:
    # DNA: merge no dataset antes de calcular KPIs
    if not dna.empty:
        chave_dna = [c for c in ['CNPJ_CIA','ANO_REF','DT_REFER','ORIGEM']
                     if c in dataset.columns and c in dna.columns]
        cols_dna = chave_dna + [c for c in ['DNA','DNA_DFC_CD','DNA_DFC_DS','DNA_DRE']
                                if c in dna.columns]
        dataset = dataset.merge(dna[cols_dna], on=chave_dna, how='left')
        logger.info("D&A integrado ao dataset | cobertura: %.0f%%",
                    dataset['DNA'].notna().mean() * 100 if 'DNA' in dataset.columns else 0)
    else:
        logger.warning("D&A vazio — EBITDA e margem_ebitda serão calculados sem D&A")

    dataset = calcular_kpis(dataset)
    logger.info("Dataset com KPIs: %d×%d", *dataset.shape)


## Etapa 4D — Feature Engineering Temporal

**O que faz:**
Cria 8 features derivadas da data de referência (`DT_REFER`). Essas features
precisam estar aqui — no Script 1 — porque são características dos dados em si,
não transformações de modelagem.

As features criadas são:

| Feature | O que representa |
|---------|-----------------|
| `ANO` | Ano do exercício (ex: 2024) |
| `TRIMESTRE` | Trimestre (1 a 4) |
| `MES` | Mês de fechamento |
| `ANO_TRIMESTRE` | Formato string '2024Q4' — útil para visualizações |
| `FLAG_FIM_ANO` | 1 se o fechamento foi em outubro, novembro ou dezembro |
| `FLAG_TRIMESTRE` | 1 se o fechamento foi em mês de fechamento trimestral (3,6,9,12) |
| `DELTA_DIAS` | Diferença em dias entre este período e o anterior da mesma empresa |
| `PERIODO_ORDINAL` | Posição temporal da empresa (1 = exercício mais antigo disponível) |

`DELTA_DIAS` e `PERIODO_ORDINAL` são especialmente úteis para o modelo ML porque
capturam a dimensão temporal dentro da série de cada empresa.


In [ ]:
def engenharia_temporal(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cria 8 features temporais a partir de DT_REFER (timezone-aware).
    Executa ANTES do salvamento — features pertencem ao dado, não à modelagem.
    """
    if df.empty or 'DT_REFER' not in df.columns:
        logger.warning("Feature engineering: DT_REFER ausente — ignorado")
        return df

    # Ordena por empresa e data para DELTA_DIAS e PERIODO_ORDINAL serem corretos
    d = df.sort_values(['CNPJ_CIA', 'ORIGEM', 'DT_REFER']).copy()
    dt = d['DT_REFER'].dt

    d['ANO']            = dt.year.astype('Int64')
    d['TRIMESTRE']      = dt.quarter.astype('Int64')
    d['MES']            = dt.month.astype('Int64')
    d['ANO_TRIMESTRE']  = dt.year.astype(str) + 'Q' + dt.quarter.astype(str)
    d['FLAG_FIM_ANO']   = dt.month.isin([10, 11, 12]).astype('Int8')
    d['FLAG_TRIMESTRE'] = dt.month.isin([3, 6, 9, 12]).astype('Int8')

    # DELTA_DIAS: dias desde o exercício anterior da mesma empresa e origem
    d['DELTA_DIAS'] = (
        d.groupby(['CNPJ_CIA', 'ORIGEM'])['DT_REFER']
        .diff().dt.days.astype('Int64')
    )
    # PERIODO_ORDINAL: posição temporal (1 = mais antigo)
    d['PERIODO_ORDINAL'] = (
        d.groupby(['CNPJ_CIA', 'ORIGEM']).cumcount() + 1
    ).astype('Int64')

    logger.info("Features temporais criadas: ANO, TRIMESTRE, MES, ANO_TRIMESTRE, "
                "FLAG_FIM_ANO, FLAG_TRIMESTRE, DELTA_DIAS, PERIODO_ORDINAL")
    return d

if not dataset.empty:
    dataset = engenharia_temporal(dataset)
    logger.info("Dataset após feature engineering: %d×%d", *dataset.shape)


## Etapa 5A — Deduplicação final e filtro de qualidade

**O que faz:**
Realiza duas operações finais de limpeza:

**Deduplicação por chave `(CNPJ_CIA, DT_REFER, ORIGEM)`:** garante que cada
par empresa/data/origem tenha exatamente um vetor de KPIs. O critério de
desempate é manter o registro com mais KPIs preenchidos — assim aproveitamos
o máximo de informação disponível.

**Filtro pós-merge outer (Correção D):** o `merge outer` pode criar linhas onde
uma empresa tem BPA mas não tem DRE para aquele período (dados parcialmente
disponíveis em um ZIP mas não em outro). Essas linhas têm KPIs quase todos `NaN`
e contaminariam o treinamento. A regra mínima é: para ser incluído, o registro
precisa ter ao menos a `DRE_3.01` (Receita Líquida) preenchida. Registros
removidos aqui são contabilizados em `AUDITORIA['alertas_merge_outer']`.


In [ ]:
if not dataset.empty:
    kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]

    # ── Deduplicação final ────────────────────────────────────────────────
    n_ant = len(dataset)
    dataset['_n_kpis'] = dataset[kpis_presentes].notna().sum(axis=1)
    dataset = (dataset
               .sort_values(['CNPJ_CIA','DT_REFER','ORIGEM','_n_kpis'],
                            ascending=[True,True,True,False])
               .drop_duplicates(subset=['CNPJ_CIA','DT_REFER','ORIGEM'], keep='first')
               .drop(columns=['_n_kpis'])
               .reset_index(drop=True))
    rem = n_ant - len(dataset)
    AUDITORIA['duplicatas_removidas'] += rem
    logger.info("Dedup final: %d → %d (-%d por CNPJ+DT_REFER+ORIGEM)", n_ant, len(dataset), rem)

    # ── PATCH 5: substituir filtro agressivo por alerta ───────────────────
    # PROBLEMA ORIGINAL (CORREÇÃO D): removia todas as linhas sem DRE_3.01.
    # Isso descartava empresas que têm DFC, BPA, BPP mas não tiveram receita
    # no período (holding pura, empresa em reestruturação, período inicial).
    # Resultado: queda de 25 → 20 empresas.
    #
    # NOVA ABORDAGEM: registrar o alerta na auditoria e manter no dataset.
    # O Script 2 (preparação) pode filtrar ou imputar conforme a necessidade.
    if 'DRE_3.01' in dataset.columns:
        mask_sem_dre = dataset['DRE_3.01'].isna()
        n_sem = mask_sem_dre.sum()
        if n_sem:
            alertas = dataset[mask_sem_dre][['CNPJ_CIA','NOME_CIA','DT_REFER','ORIGEM']].to_dict('records')
            AUDITORIA['alertas_merge_outer'].extend(alertas[:50])
            logger.warning(
                "PATCH 5: %d linhas sem DRE_3.01 (Receita) MANTIDAS no dataset "
                "(eram removidas no V4 — causa da queda 25→20 empresas). "
                "Verificar: %s",
                n_sem,
                [(a['NOME_CIA'],str(a['DT_REFER'])[:10]) for a in alertas[:5]]
            )

    logger.info("Dataset consolidado final: %d × %d | %d empresas",
                *dataset.shape, dataset['CNPJ_CIA'].nunique())
    # Distribuição por origem
    if 'ORIGEM' in dataset.columns:
        for orig, cnt in dataset['ORIGEM'].value_counts().items():
            logger.info("  %-5s: %d linhas (%.0f%%)", orig, cnt, 100*cnt/len(dataset))


## Etapa 5B — Análise Exploratória de Dados (9 Blocos)

**O que faz:**
Realiza uma análise descritiva estruturada do dataset final antes de salvá-lo.
Os 9 blocos cobrem diferentes dimensões do dado e servem tanto para validação
quanto para a seção de metodologia do TCC.

Cada bloco é uma célula separada para facilitar a execução independente e a
leitura dos resultados no Jupyter.


In [ ]:
if dataset.empty:
    logger.warning("Dataset vazio — EDA ignorada.")
else:
    kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]

    print("=" * 70)
    print("  BLOCO 1 — Visão Geral do Dataset")
    print("=" * 70)
    print(f"  Linhas        : {len(dataset):,}")
    print(f"  Colunas       : {dataset.shape[1]}")
    print(f"  Empresas      : {dataset['NOME_CIA'].nunique()} / 25")
    print(f"  Setores       : {dataset['SETOR'].nunique()}")
    if 'ANO' in dataset.columns:
        print(f"  Anos cobertos : {sorted(dataset['ANO'].dropna().astype(int).unique())}")
    if 'ORIGEM' in dataset.columns:
        print(f"  Por origem    : {dataset['ORIGEM'].value_counts().to_dict()}")
    print(f"  Nulos global  : {dataset.isnull().mean().mean():.1%}")
    print(f"  Erros de data : {len(AUDITORIA['erros_data'])}")
    print(f"  Datas futuras : {len(AUDITORIA['datas_futuras'])}")
    print(f"  Períodos irr. : {len(AUDITORIA['periodos_irregulares'])}")


In [ ]:
if not dataset.empty:
    print("BLOCO 2 — Cobertura Temporal por Empresa")
    cob = (dataset.groupby(['NOME_CIA','ORIGEM'])
           .agg(Primeiro=('ANO','min'), Último=('ANO','max'),
                Períodos=('ANO','count'), Delta_médio=('DELTA_DIAS','mean'))
           .round({'Delta_médio':0})
           .sort_values(['NOME_CIA','ORIGEM']))
    print(cob.to_string())


In [ ]:
if not dataset.empty:
    print("BLOCO 3 — Inventário de Contas por Demonstrativo")
    for prefixo in ['DRE','BPA','BPP','DFC']:
        cols = [c for c in dataset.columns if c.startswith(f'{prefixo}_')]
        if not cols: continue
        cob = dataset[cols].notna().mean().sort_values(ascending=False)
        print(f"  {prefixo}: {len(cols)} contas | Top-5: {list(cob.head(5).index)}")


In [ ]:
if not dataset.empty:
    print("BLOCO 4 — Qualidade dos Dados (KPIs)")
    kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]
    nulos = dataset[kpis_presentes].isnull().mean().sort_values(ascending=False)
    for kpi, v in nulos.items():
        s = "❌" if v > 0.5 else ("⚠️" if v > 0.2 else "✅")
        print(f"  {s} {kpi:<25} {v:.0%}")
    df_num = dataset[kpis_presentes].dropna()
    if not df_num.empty:
        z = np.abs(stats.zscore(df_num, axis=0, nan_policy='omit'))
        print(f"\n  Outliers extremos (|z|>5): {(z > 5).sum().sum()}")


In [ ]:
if not dataset.empty:
    print("BLOCO 5 — Estatísticas Descritivas dos KPIs")
    desc = dataset[kpis_presentes].describe().T
    print(desc[['count','mean','std','min','50%','max']].round(3).to_string())


In [ ]:
if not dataset.empty:
    print("BLOCO 6 — Targets Principais por Setor (mediana, R$ mil)")
    targets = [c for c in ['DRE_3.01','DRE_3.11','EBITDA'] if c in dataset.columns]
    if targets:
        print(dataset.groupby('SETOR')[targets].median().round(0).to_string())


In [ ]:
if not dataset.empty:
    print("BLOCO 7 — Top-10 Correlações de Pearson com Receita Líquida")
    if 'DRE_3.01' in dataset.columns:
        corr = dataset[kpis_presentes + ['DRE_3.01']].corr()['DRE_3.01'].drop('DRE_3.01')
        print(corr.sort_values(key=abs, ascending=False).head(10).to_string())


In [ ]:
if not dataset.empty:
    print("BLOCO 8 — Evolução Temporal de Receita por Setor (DFP apenas)")
    df_plot = dataset[dataset.get('ORIGEM', pd.Series('DFP')) == 'DFP'] if 'ORIGEM' in dataset.columns else dataset
    if 'DRE_3.01' in df_plot.columns and 'ANO' in df_plot.columns:
        evol = df_plot.groupby(['SETOR','ANO'])['DRE_3.01'].median().unstack('SETOR').dropna(how='all')
        print(evol.round(0).to_string())
        fig, ax = plt.subplots(figsize=(12,5))
        evol.plot(ax=ax, marker='o')
        ax.set_title('Receita Líquida Mediana por Setor — DFP (R$ mil)')
        ax.set_xlabel('Ano'); ax.set_ylabel('R$ mil')
        plt.tight_layout()
        plt.savefig(PASTA_SAIDA / 'evol_receita_setor.png', dpi=150)
        plt.show()


In [ ]:
if not dataset.empty:
    print("BLOCO 9 — Inventário Final de KPIs para Modelagem")
    kpis_ok  = [k for k in kpis_presentes if dataset[k].notna().mean() > 0.5]
    kpis_exc = [k for k in kpis_presentes if k not in kpis_ok]
    print(f"  ✅ KPIs com >50% cobertura ({len(kpis_ok)}):")
    for k in kpis_ok:
        print(f"      {k:<25} {dataset[k].notna().mean():.0%}")
    print(f"  ❌ KPIs excluídos ({len(kpis_exc)}):")
    for k in kpis_exc:
        print(f"      {k:<25} {dataset[k].notna().mean():.0%}")


## Etapa 6 — Persistência: Parquet + CSV + Auditoria JSON

**O que faz:**
Salva o dataset final em três arquivos:

**Parquet (formato principal):** usa compressão snappy e preserva todos os
tipos de dados (datetime com timezone, Int64, float64). É cerca de 10x mais
rápido para ler que CSV e ocupa menos espaço. O Script 2 sempre lerá o Parquet.

**CSV (opcional):** salvo com encoding `utf-8-sig` para compatibilidade com
Excel no Windows. É o formato de fallback caso o ambiente não suporte Parquet.

**Relatório de auditoria JSON:** documenta todas as métricas do pipeline —
quantas linhas foram lidas, quantas descartadas, erros de data encontrados,
duplicatas removidas, cobertura de cada KPI. Esse arquivo é a rastreabilidade
completa do processamento, algo essencial para um TCC de qualidade.

O Parquet preserva a coluna `ORIGEM` (`DFP` ou `ITR`), permitindo que o
Script 2 filtre conforme necessário sem precisar reprocessar os dados brutos.


In [ ]:
if not dataset.empty:
    # Parquet — formato principal para o pipeline ML
    cam_pq = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
    dataset.to_parquet(cam_pq, index=False, compression='snappy', engine='pyarrow')
    logger.info("Parquet: %s | %d KB", cam_pq, cam_pq.stat().st_size // 1024)

    # CSV — compatibilidade Excel/Windows
    cam_csv = PASTA_SAIDA / 'dataset_cvm_consolidado.csv'
    dataset.to_csv(cam_csv, index=False, encoding='utf-8-sig')
    logger.info("CSV    : %s | %d KB", cam_csv, cam_csv.stat().st_size // 1024)

    # Relatório de auditoria — rastreabilidade completa do pipeline
    relatorio = {
        'timestamp'                  : AGORA_LOCAL.isoformat(),
        'modo_rigoroso'              : MODO_RIGOROSO,
        'pasta_dfp'                  : str(PASTA_DFP),
        'pasta_itr'                  : str(PASTA_ITR),
        'zips_processados'           : AUDITORIA['zips_processados'],
        'linhas_lidas_total'         : int(AUDITORIA['linhas_lidas_total']),
        'linhas_filtradas_anchor'    : int(AUDITORIA['linhas_filtradas_anchor']),
        'registros_descartados'      : int(AUDITORIA['registros_descartados']),
        'duplicatas_removidas'       : int(AUDITORIA['duplicatas_removidas']),
        'erros_data_total'           : len(AUDITORIA['erros_data']),
        'datas_futuras_removidas'    : len(AUDITORIA['datas_futuras']),
        'inconsistencias_ano'        : len(AUDITORIA['inconsistencias_ano']),
        'periodos_irregulares'       : len(AUDITORIA['periodos_irregulares']),
        'alertas_merge_outer'        : len(AUDITORIA['alertas_merge_outer']),
        'erros_data_amostra'         : AUDITORIA['erros_data'][:20],
        'dataset_shape'              : list(dataset.shape),
        'dataset_empresas'           : int(dataset['NOME_CIA'].nunique()),
        'dataset_anos'               : (sorted(dataset['ANO'].dropna().astype(int).unique().tolist())
                                        if 'ANO' in dataset.columns else []),
        'dataset_origens'            : (dataset['ORIGEM'].value_counts().to_dict()
                                        if 'ORIGEM' in dataset.columns else {}),
        'kpis_cobertura'             : {
            k: round(float(dataset[k].notna().mean()), 4)
            for k in LISTA_KPIS if k in dataset.columns
        },
    }
    cam_audit = PASTA_SAIDA / 'auditoria_processamento.json'
    with open(cam_audit, 'w', encoding='utf-8') as f:
        json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)
    logger.info("Auditoria: %s", cam_audit)

    # Resumo final no terminal
    print("\n" + "═" * 70)
    print("  RESUMO FINAL — Script 1 (cvm_processamento) — V5")
    print("═" * 70)
    print(f"  Dataset        : {dataset.shape[0]:,} linhas × {dataset.shape[1]} colunas")
    print(f"  Empresas       : {dataset['NOME_CIA'].nunique()} / 25")
    if 'ANO' in dataset.columns:
        print(f"  Anos           : {relatorio['dataset_anos']}")
    if 'ORIGEM' in dataset.columns:
        print(f"  Por origem     : {relatorio['dataset_origens']}")
    print(f"  Erros de data  : {len(AUDITORIA['erros_data'])}")
    print(f"  Datas futuras  : {len(AUDITORIA['datas_futuras'])}")
    print(f"  Duplicatas rem.: {AUDITORIA['duplicatas_removidas']}")
    print(f"  Parquet        : {cam_pq}")
    print(f"  CSV            : {cam_csv}")
    print(f"  Auditoria      : {cam_audit}")
    print("═" * 70)
    print("  ✅ Pronto para o Script 2 (02_cvm_preparacao.ipynb)")
    print("═" * 70)

    # Cobertura de KPIs no resumo final
    print("\n  Cobertura de KPIs:")
    for kpi in LISTA_KPIS:
        if kpi in dataset.columns:
            cob = dataset[kpi].notna().mean()
            s = "✅" if cob >= 0.7 else ("⚠️" if cob >= 0.5 else "❌")
            print(f"    {s} {kpi:<25} {cob:.0%}")
else:
    logger.error("Dataset vazio — verifique TCC_dados/DFP/ e TCC_dados/ITR/")
